## Solving the Poisson Problem on a 4 dimensional semistructured P2 Fes

This Notebook is very similar to the P1 notebook. In general a more interesting application of the P2 element is in the [Navier-Stokes](/demos/linearised_Navier_Stokes/taylor-Hood.ipynb) Taylor-Hood discretization

In [1]:
from import_hack import *
from methodsnm.mesh_4d import *
from methodsnm.visualize import *
import math
import numpy as np
from numpy import exp
from methodsnm.vectorspace import *
from methodsnm.fes import *
from netgen.csg import unit_cube
from ngsolve import Mesh,VOL,specialcf
from numpy import cos, sin,pi

source module for methodsNM imported.


Constructing a mesh

In [2]:
m = 2
T = 2
epsi = 0.1
ngmesh = Mesh(unit_cube.GenerateMesh(maxh=0.5))
mesh = UnstructuredHypertriangleMesh(T,ngmesh)
def list_diff(a, b):
    """Entfernt alle Elemente aus Liste a, die in Liste b enthalten sind."""
    return [x for x in a if x not in b]

In [3]:
from methodsnm.forms import *
from methodsnm.formint import *
V = P2_Hypertriangle_Space(mesh)
V1 = P1_Hypertriangle_Space(mesh)
print("Number of dofs for P1Fes :",V1.ndof , V.nv ,V.nedges)
print("Number of dofs for P2Fes :",V.ndof , "As expected it is much larger than for the P1 element")
print(len(V.boundary_dofs()), "also their are more boundary dofs than in P1", len(mesh.bndry_vertices))
#print(mesh.hypercell2edge)
print(len(mesh.edges))
print(len(mesh.boundary_edges),len(V.boundary_vertices()))
print(list(mesh.boundary_edges.values()))


Number of dofs for P1Fes : 78 78 574
Number of dofs for P2Fes : 652 As expected it is much larger than for the P1 element
120 also their are more boundary dofs than in P1 72
574
48 72
[1, 3, 8, 10, 16, 17, 23, 24, 29, 31, 36, 38, 44, 45, 50, 52, 242, 249, 256, 263, 270, 277, 284, 291, 478, 479, 480, 481, 482, 483, 484, 485, 486, 487, 488, 489, 490, 491, 492, 493, 494, 495, 496, 497, 498, 499, 500, 501]


In [4]:
print(V.boundary_dofs())

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 79, 81, 86, 88, 94, 95, 101, 102, 107, 109, 114, 116, 122, 123, 128, 130, 320, 327, 334, 341, 348, 355, 362, 369, 556, 557, 558, 559, 560, 561, 562, 563, 564, 565, 566, 567, 568, 569, 570, 571, 572, 573, 574, 575, 576, 577, 578, 579]


In [5]:
import numpy as np
np.set_printoptions(threshold=np.inf)
u_ex = lambda x: cos(pi*x[0])*cos(pi*x[1])*cos(pi*x[2])*cos(pi*x[3])
u_h = FEFunction(V)
u_h._set_P2(u_ex,boundary=True)
#print(u_h.vector,np.count_nonzero(u_h.vector))
boundary_dofs = V.boundary_dofs()
def contains_0_or_1(lst):
    return any(x in (0, 1) for x in lst)

for dof in boundary_dofs:
    if dof >= V.nv:
        eid = dof - V.nv
        v1, v2 = mesh.edges[eid]
        midpoint = 0.5*(mesh.points[v1] + mesh.points[v2])
        if u_ex(midpoint) != u_h.vector[dof]:
            print("Error at midpoint of edge", eid)
            print("Midpoint:", midpoint)
            print("Exact value:", u_ex(midpoint))
            print("Computed value:", u_h.vector[dof])
        if not contains_0_or_1(midpoint):
            print(midpoint, "with value", mesh.points[v1] ,mesh.points[v2])

Assembling the Biliniearform and the Linearform for P2 elements, and treating the boundary conditions accordingly

In [6]:
from numpy import cos, sin,pi
BF = BilinearForm(V)                      
BF += LaplaceIntegral()
BF.assemble()

u_ex = lambda x: cos(pi*x[0])*cos(pi*x[1])*cos(pi*x[2])*cos(pi*x[3])
f    = lambda x: 4*pi**2*cos(pi*x[0])*cos(pi*x[1])*cos(pi*x[2])*cos(pi*x[3])
f = GlobalFunction(f, mesh=mesh)

LFs = LinearForm(V)
LFs += SourceIntegral(f)
LFs.assemble()

In [7]:
all_dofs = list(range(V.ndof))
boundary_dofs = V.boundary_dofs()
freedofs = list_diff(all_dofs, boundary_dofs)

u_h = FEFunction(V)
u_h._set_P2(u_ex,boundary=True)

res =  LFs.vector - BF.matrix @ u_h.vector 
from methodsnm.solver import solve_on_freedofs
u_h.vector += solve_on_freedofs(BF.matrix, res, freedofs)

uex = FEFunction(V)
uex._set_P2(u_ex)
res = BF.matrix @ uex.vector - LFs.vector


In [8]:
from methodsnm.forms import compute_difference_L2
l2diff = compute_difference_L2(u_h, GlobalFunction(u_ex,mesh=mesh), mesh, intorder = 3)

print("l2diff =", l2diff)

l2diff = 0.045872056099501325


## Final remarks
So if you compare it to the P1 element you get better results, if you take the same mesh construction and in addition will also see a better convergence rate.

In [ ]:
from ngsolve import Mesh,VOL,specialcf
from numpy import cos
h = 0.5
m = 2
uex =  GlobalFunction(lambda x: cos(pi*x[0])*cos(pi*x[1])*cos(pi*x[2])*cos(pi*x[3]), mesh = mesh)
mesh_list = []
for i in range(3):
    ngmesh = Mesh(unit_cube.GenerateMesh(maxh=h))
    mesh_list.append(UnstructuredHypertriangleMesh(m,ngmesh))
    m = 2*m
    h = h/2

for mesh in mesh_list:
    V = P2_Hypertriangle_Space(mesh)
    BF = BilinearForm(V)                      
    BF += LaplaceIntegral()
    BF.assemble()

    u_ex = lambda x: cos(pi*x[0])*cos(pi*x[1])*cos(pi*x[2])*cos(pi*x[3])
    f    = lambda x: 4*pi**2*cos(pi*x[0])*cos(pi*x[1])*cos(pi*x[2])*cos(pi*x[3])
    f = GlobalFunction(f, mesh=mesh)

    LFs = LinearForm(V)
    LFs += SourceIntegral(f)
    LFs.assemble()

    all_dofs = list(range(V.ndof))
    boundary_dofs = V.boundary_dofs()
    freedofs = list_diff(all_dofs, boundary_dofs)

    u_h = FEFunction(V)
    u_h._set_P2(u_ex,boundary=True)

    res =  LFs.vector - BF.matrix @ u_h.vector 
    from methodsnm.solver import solve_on_freedofs
    u_h.vector += solve_on_freedofs(BF.matrix, res, freedofs)

    uex = FEFunction(V)
    uex._set_P2(u_ex)
    res = BF.matrix @ uex.vector - LFs.vector
    
    l2diff = compute_difference_L2(u_h, GlobalFunction(u_ex,mesh=mesh), mesh, intorder = 3)
    print("l2diff =", l2diff)

l2diff = 0.045872056099501325
l2diff = 0.007445527380015314
